In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    f1_score,
    precision_score,
    recall_score
)

import matplotlib.pyplot as plt



# ==========================================================
# Mask-Guided Variable Window Pooling (same as your code)
# ==========================================================
class MaskGuidedVariablePooling(nn.Module):
    def __init__(self, K, tau=1.0):
        super().__init__()
        self.K = K
        self.tau = tau

        self.h_mlp = nn.Sequential(
            nn.LazyLinear(64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

        self.last_hk = None  # [B]
        self.last_m  = None  # [B,W]

    def forward(self, x):
        # x: [B, C, H, W]
        B, C, H, W = x.shape
        device = x.device

        pooled = x.mean(dim=[2,3])          # [B, C]
        h_k = self.h_mlp(pooled).squeeze(1) # [B]

        t = torch.linspace(0.0, 1.0, W, device=device).unsqueeze(0)  # [1,W]
        m = torch.sigmoid(self.tau * (h_k.unsqueeze(1) - t))         # [B,W]

        self.last_hk = h_k
        self.last_m  = m

        s = torch.relu(m[:, :-1] - m[:, 1:])  # [B,W-1]
        eps = 1e-8
        p = (s + eps) / (s.sum(dim=1, keepdim=True) + eps)  # [B,W-1]
        cdf = torch.cumsum(p, dim=1)                         # [B,W-1]
        cdf[:, -1] = 1.0

        quantiles = torch.linspace(0, 1, self.K+1, device=device)[1:-1]  # [K-1]
        cuts = []
        for q in quantiles:
            cut = torch.argmax((cdf >= q).int(), dim=1)  # [B]
            cuts.append(cut)
        cuts = torch.stack(cuts, dim=1) if len(cuts) else torch.empty(B,0,dtype=torch.long,device=device)

        outputs = []
        prev = torch.zeros(B, dtype=torch.long, device=device)
        idx = torch.arange(W, device=device).unsqueeze(0)  # [1,W]

        for r in range(self.K):
            if r < self.K-1:
                curr = cuts[:, r]
            else:
                curr = torch.full((B,), W-1, device=device, dtype=torch.long)

            seg_mask = ((idx >= prev.unsqueeze(1)) & (idx <= curr.unsqueeze(1))).float()
            weights = seg_mask * m
            weights = weights / (weights.sum(dim=1, keepdim=True) + eps)

            pooled_seg = (x * weights.unsqueeze(1).unsqueeze(2)).sum(dim=3)  # sum over W
            outputs.append(pooled_seg)
            prev = curr + 1

        return torch.stack(outputs, dim=3)  # [B,C,H,K]


# ==========================================================
# Regularization on hk
# ==========================================================
def hk_class_regularization(hk, y, eps=1e-8):
    unique = torch.unique(y)
    means = []
    intra = hk.new_tensor(0.0)
    valid_classes = 0

    for c in unique:
        mask = (y == c)
        if mask.sum() >= 2:
            hk_c = hk[mask]
            means.append(hk_c.mean())
            intra = intra + hk_c.var(unbiased=False)
            valid_classes += 1
        elif mask.sum() == 1:
            means.append(hk[mask].mean())
            valid_classes += 1

    L_intra = intra / (valid_classes + eps) if valid_classes > 0 else hk.new_tensor(0.0)

    if len(means) >= 2:
        means = torch.stack(means)
        diff = means.unsqueeze(0) - means.unsqueeze(1)
        L_inter = diff.pow(2).mean()
    else:
        L_inter = hk.new_tensor(0.0)

    return L_intra, L_inter


# ==========================================================
# Model (dynamic sizes)
# ==========================================================
class Model(nn.Module):
    def __init__(self, n_features: int, d_hist: int, n_classes: int = 2, tau: float = 1.0):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, (1, 3))
        self.pool1 = MaskGuidedVariablePooling(K=7, tau=tau)

        self.conv2 = nn.Conv2d(16, 16, (1, 2))
        self.pool2 = MaskGuidedVariablePooling(K=5, tau=tau)

        self.flatten = nn.Flatten()

        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_features, d_hist)
            dummy = torch.relu(self.conv1(dummy))
            dummy = self.pool1(dummy)
            dummy = torch.relu(self.conv2(dummy))
            dummy = self.pool2(dummy)
            flatten_size = dummy.numel()

        self.fc = nn.Sequential(
            nn.Linear(flatten_size, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool1(x)
        x = torch.relu(self.conv2(x))
        x = self.pool2(x)
        x = self.flatten(x)
        return self.fc(x)


# =========================
# Dataset (same as training)
# =========================
class SwatSeqFromFullDataset(Dataset):
    def __init__(self, X_full, y_full, indices, d_hist=20, repeat_pattern=True):
        self.X = X_full.astype(np.float32, copy=False)
        self.y = y_full.astype(np.int64, copy=False)
        self.idx = indices.astype(np.int64, copy=False)
        self.W = int(d_hist)
        self.repeat_pattern = bool(repeat_pattern)

    def __len__(self):
        return len(self.idx)

    def _make_matrix(self, i: int) -> np.ndarray:
        x_i = self.X[i]
        F = x_i.shape[0]
        W = self.W

        seq = [x_i]
        j = i - 1
        while j >= 0 and len(seq) < W:
            seq.append(self.X[j])
            j -= 1

        seq = np.stack(seq, axis=1)  # [F, L]

        if seq.shape[1] < W:
            if self.repeat_pattern:
                num_repeats = W // seq.shape[1]
                rem = W % seq.shape[1]
                out = np.tile(seq, (1, num_repeats))
                if rem > 0:
                    out = np.hstack([out, seq[:, :rem]])
            else:
                pad_cols = W - seq.shape[1]
                pad = np.tile(x_i.reshape(F, 1), (1, pad_cols))
                out = np.hstack([seq, pad])
        else:
            out = seq[:, :W]

        return out.astype(np.float32, copy=False)

    def __getitem__(self, k: int):
        i = int(self.idx[k])
        mat = self._make_matrix(i)                 # [F,W]
        lab = int(self.y[i])                       # scalar
        return torch.from_numpy(mat), torch.tensor(lab, dtype=torch.long)


# =======================================
# Model definitions (same as training)
# =======================================
class MaskGuidedVariablePooling(nn.Module):
    def __init__(self, K, tau=1.0):
        super().__init__()
        self.K = K
        self.tau = tau

        self.h_mlp = nn.Sequential(
            nn.LazyLinear(64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

        self.last_hk = None
        self.last_m  = None

    def forward(self, x):
        B, C, H, W = x.shape
        device = x.device

        pooled = x.mean(dim=[2,3])
        h_k = self.h_mlp(pooled).squeeze(1)

        t = torch.linspace(0.0, 1.0, W, device=device).unsqueeze(0)
        m = torch.sigmoid(self.tau * (h_k.unsqueeze(1) - t))

        self.last_hk = h_k
        self.last_m  = m

        s = torch.relu(m[:, :-1] - m[:, 1:])
        eps = 1e-8
        p = (s + eps) / (s.sum(dim=1, keepdim=True) + eps)
        cdf = torch.cumsum(p, dim=1)
        cdf[:, -1] = 1.0

        quantiles = torch.linspace(0, 1, self.K+1, device=device)[1:-1]
        cuts = []
        for q in quantiles:
            cut = torch.argmax((cdf >= q).int(), dim=1)
            cuts.append(cut)
        cuts = torch.stack(cuts, dim=1) if len(cuts) else torch.empty(B,0,dtype=torch.long,device=device)

        outputs = []
        prev = torch.zeros(B, dtype=torch.long, device=device)
        idx = torch.arange(W, device=device).unsqueeze(0)

        for r in range(self.K):
            if r < self.K-1:
                curr = cuts[:, r]
            else:
                curr = torch.full((B,), W-1, device=device, dtype=torch.long)

            seg_mask = ((idx >= prev.unsqueeze(1)) & (idx <= curr.unsqueeze(1))).float()
            weights = seg_mask * m
            weights = weights / (weights.sum(dim=1, keepdim=True) + eps)

            pooled_seg = (x * weights.unsqueeze(1).unsqueeze(2)).sum(dim=3)
            outputs.append(pooled_seg)
            prev = curr + 1

        return torch.stack(outputs, dim=3)


class Model(nn.Module):
    def __init__(self, n_features: int, d_hist: int, n_classes: int = 2, tau: float = 1.0):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, (1,3))
        self.pool1 = MaskGuidedVariablePooling(K=7, tau=tau)
        self.conv2 = nn.Conv2d(16, 16, (1,2))
        self.pool2 = MaskGuidedVariablePooling(K=5, tau=tau)
        self.flatten = nn.Flatten()

        with torch.no_grad():
            dummy = torch.zeros(1,1,n_features,d_hist)
            dummy = torch.relu(self.conv1(dummy))
            dummy = self.pool1(dummy)
            dummy = torch.relu(self.conv2(dummy))
            dummy = self.pool2(dummy)
            flatten_size = dummy.numel()

        self.fc = nn.Sequential(
            nn.Linear(flatten_size, 64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,n_classes)
        )

    def forward(self,x):
        x = torch.relu(self.conv1(x))
        x = self.pool1(x)
        x = torch.relu(self.conv2(x))
        x = self.pool2(x)
        x = self.flatten(x)
        return self.fc(x)


# =========================
# Split builders (same)
# =========================
def build_chronological_split_indices(N: int, split_ratio: float = 0.7):
    cut = int(N * split_ratio)
    train_idx = np.arange(0, cut, dtype=np.int64)
    test_idx  = np.arange(cut, N, dtype=np.int64)
    return train_idx, test_idx

def build_stratified_random_split_indices(y: np.ndarray, test_size: float = 0.3, seed: int = 843):
    all_idx = np.arange(len(y), dtype=np.int64)
    tr, te = train_test_split(all_idx, test_size=test_size, random_state=seed, stratify=y)
    return np.sort(tr), np.sort(te)


# =========================
# Inference helpers
# =========================
@torch.no_grad()
def predict_proba_attack(model, loader, device):
    """
    Returns:
      y_true: [N]
      p_attack: [N] probability of class 1
      y_pred: [N] argmax prediction
    """
    model.eval()
    ys = []
    ps = []
    preds = []
    softmax = nn.Softmax(dim=1)

    for xb, yb in loader:
        xb = xb.unsqueeze(1).to(device, non_blocking=True)
        yb = yb.cpu().numpy()

        logits = model(xb)
        prob = softmax(logits).detach().cpu().numpy()   # [B,2]
        p1 = prob[:, 1]
        yhat = prob.argmax(axis=1)

        ys.append(yb)
        ps.append(p1)
        preds.append(yhat)

    y_true = np.concatenate(ys)
    p_attack = np.concatenate(ps)
    y_pred = np.concatenate(preds)
    return y_true, p_attack, y_pred


def find_best_threshold(y_true, p_attack):
    """
    Sweep thresholds for class 1 and pick the one maximizing F1 on Attack.
    """
    thresholds = np.linspace(0.0, 1.0, 501)
    best = {"thr": 0.5, "f1": -1.0, "prec": None, "rec": None}
    for thr in thresholds:
        y_hat = (p_attack >= thr).astype(int)
        f1 = f1_score(y_true, y_hat, pos_label=1, zero_division=0)
        if f1 > best["f1"]:
            best["f1"] = f1
            best["thr"] = thr
            best["prec"] = precision_score(y_true, y_hat, pos_label=1, zero_division=0)
            best["rec"] = recall_score(y_true, y_hat, pos_label=1, zero_division=0)
    return best


def full_report(name, y_true, p_attack, y_pred, plot_curves=True):
    print(f"\n==================== {name} ====================")
    print("Counts:", {0: int((y_true==0).sum()), 1: int((y_true==1).sum())})

    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    print("\nConfusion matrix [rows=true, cols=pred] (0=Normal,1=Attack):\n", cm)

    print("\nClassification report (argmax, thr=0.5):")
    print(classification_report(y_true, y_pred, target_names=["Normal","Attack"], digits=4, zero_division=0))

    bal_acc = balanced_accuracy_score(y_true, y_pred)
    print(f"Balanced accuracy: {bal_acc:.4f}")

    # AUC metrics (need both classes present)
    if len(np.unique(y_true)) == 2:
        roc_auc = roc_auc_score(y_true, p_attack)
        pr_auc = average_precision_score(y_true, p_attack)
        print(f"ROC-AUC (Attack=1): {roc_auc:.4f}")
        print(f"PR-AUC  (Attack=1): {pr_auc:.4f}")
    else:
        print("ROC-AUC/PR-AUC skipped: only one class present in y_true.")

    best = find_best_threshold(y_true, p_attack)
    print("\nBest threshold for Attack (max F1):")
    print(f"  thr={best['thr']:.3f} | F1={best['f1']:.4f} | P={best['prec']:.4f} | R={best['rec']:.4f}")

    if plot_curves and len(np.unique(y_true)) == 2:
        fpr, tpr, _ = roc_curve(y_true, p_attack)
        prec, rec, _ = precision_recall_curve(y_true, p_attack)

        plt.figure()
        plt.plot(fpr, tpr)
        plt.plot([0,1],[0,1])
        plt.xlabel("FPR")
        plt.ylabel("TPR")
        plt.title(f"ROC curve - {name}")
        plt.show()

        plt.figure()
        plt.plot(rec, prec)
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(f"PR curve - {name}")
        plt.show()


# =========================
# MAIN notebook evaluation
# =========================
DATA_DIR = "./preprocessed_swat/"
MODELS_DIR = "./swat_model/"
D_HIST = 20
BS = 512
TAU = 1.0
REPEAT_PATTERN = True

X_global = np.load(os.path.join(DATA_DIR, "X_global.npy"))
y_global = np.load(os.path.join(DATA_DIR, "y_global.npy"))
N, F = X_global.shape
print("Loaded X_global:", X_global.shape, "| y_global:", y_global.shape)
print("y counts:", {0: int((y_global==0).sum()), 1: int((y_global==1).sum())})

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --- Build test indices for both settings (same as training)
_, te_chrono = build_chronological_split_indices(N, split_ratio=0.7)
_, te_rand = build_stratified_random_split_indices(y_global, test_size=0.3, seed=843)

# --- Build loaders
test_ds_chrono = SwatSeqFromFullDataset(X_global, y_global, te_chrono, d_hist=D_HIST, repeat_pattern=REPEAT_PATTERN)
test_ds_rand   = SwatSeqFromFullDataset(X_global, y_global, te_rand,   d_hist=D_HIST, repeat_pattern=REPEAT_PATTERN)

test_loader_chrono = DataLoader(test_ds_chrono, batch_size=BS, shuffle=False, pin_memory=True)
test_loader_rand   = DataLoader(test_ds_rand,   batch_size=BS, shuffle=False, pin_memory=True)

# --- Load and evaluate chrono model
model_chrono = Model(n_features=F, d_hist=D_HIST, n_classes=2, tau=TAU).to(device)
model_chrono.load_state_dict(torch.load(os.path.join(MODELS_DIR, "model_swat_chrono.pth"), map_location=device))
y_true, p_attack, y_pred = predict_proba_attack(model_chrono, test_loader_chrono, device)
full_report("SWaT - Chronological split (TEST)", y_true, p_attack, y_pred, plot_curves=True)

# --- Load and evaluate random stratified model
model_rand = Model(n_features=F, d_hist=D_HIST, n_classes=2, tau=TAU).to(device)
model_rand.load_state_dict(torch.load(os.path.join(MODELS_DIR, "best_model_swat_random_stratified.pth"), map_location=device))
y_true, p_attack, y_pred = predict_proba_attack(model_rand, test_loader_rand, device)
full_report("SWaT - Random stratified split (TEST)", y_true, p_attack, y_pred, plot_curves=True)